In [1]:
# 1. Install required packages (only needed once, re-run if venv is fresh)
#    Pre-built wheels only - works with the regular CPython 3.13 inside ./myenv
%pip install pandas pyarrow nltk regex tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 2. Load the Gujarati corpus that was converted to Parquet in the previous step
#    The full file is 21.8M rows / 5.5 GB, so we use pyarrow's streaming reader
#    instead of loading everything into memory at once.
import os

input_file  = r"E:\STUDY\NLP\Language\gu.txt"
output_file = r"C:\Users\91942\OneDrive\Desktop\NLP\Lab 1\gujarati.parquet"


### Raw data in txt formate

In [3]:
# 3. Inspect the first 30 lines of the raw text file
with open(input_file, "r", encoding="utf-8") as f:
    for i in range(30):
        line = f.readline()
        print(line.strip())

આ વીડિયો જુઓ: ઊંઝા માર્કેટયાર્ડ આજથી 25 જુલાઈ સુધી બંધ

મિથેનોલ આવ્યો ક્યાંથી?

આખરે ત્રણ રાજ્યોમાં મળેલ હાર પર કોંગ્રેસ અધ્યક્ષ રાહુલ ગાંધી દ્વારા પ્રથમ પ્રતિક્રિયા આપવામાં આવી છે. તેમણે કહ્યું કે, ત્રિપુરા, નાગાલેન્ડ અને મેઘાલયમાં લોકોના જનાદેશનો સ્વાગત કરીએ છે અને આ ક્ષેત્રના લોકોનો વિશ્વાસ ફરીથી જીતીવા માટે પ્રતિબદ્ધ છીએ.

આ આંકડો માટે, અને વજન ઘટાડવા માટે પ્રકાશનનો દિવસ વિતાવવો ઉપયોગી છે, ઉદાહરણ તરીકે, અઠવાડિયામાં એક વખત. તમારા માટે એક વિકલ્પ પસંદ કરો જે અગવડતાને કારણે નહીં કરે. સૌથી વધુ લોકપ્રિય કીફિર પર અનલોડ છે.

આ ઠેકાઓ પરથી લીમડી તેમજ ઝાલોદના બૂટલેગરો વિદેશી દારૃનો મોટાપાયે જથ્થો ખરીદી મહિસાગર, હાલોલ, ગોધરા, આણંદ, નડિયાદ અને વડોદરા શહેર-જિલ્લામાં ઠાલવે છે. આબુથી આવતો દારૃનો જથ્થો સાંચોર તેમજ પાલનપુર થઈ મહેસાણામાં રીટા અને વિરસિંહને ત્યાં ઉતારવામાં આવે છે, જ્યાંથી અમદાવાદ તેમજ ઉત્તર ગુજરાતમાં દારૃ સપ્લાઈ થાય છે. રાજસ્થાનના બિચ્છુવાડાથી શામળાજી બોર્ડર થઈ હિંમતનગર, ગાંધીનગર અને અમદાવાદમાં બૂટલેગર સુનિલ, વિનોદ, દિલીપ અને રબારી દ્વારા મોટાપાયે દારૃનો જથ્થો ઠલવાય છે.

કેન્દ્રીય મંત

### Convert into parquet

In [4]:
if not os.path.exists(output_file):
    import pandas as pd
    with open(input_file, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
    pd.DataFrame({"text": lines}).to_parquet(output_file, index=False)
    print(f"Converted {len(lines):,} lines to '{output_file}'")
else:
    print(f"Parquet already exists: {output_file}")

Parquet already exists: C:\Users\91942\OneDrive\Desktop\NLP\Lab 1\gujarati.parquet


### load sample of 1st 50,000 

In [5]:
import pyarrow.parquet as pq
pf = pq.ParquetFile(output_file)
print(f"Total rows in parquet: {pf.metadata.num_rows:,}")
SAMPLE = 50_000
first_batch = next(pf.iter_batches(batch_size=SAMPLE, columns=["text"]))
import pandas as pd
df = first_batch.to_pandas()
print(f"Loaded sample of {len(df):,} rows for processing")
df.head()

Total rows in parquet: 21,798,320
Loaded sample of 50,000 rows for processing


,text
0,આ વીડિયો જુઓ: ઊંઝા માર્કેટયાર્ડ આજથી 25 જુલાઈ ...
1,મિથેનોલ આવ્યો ક્યાંથી?
2,આખરે ત્રણ રાજ્યોમાં મળેલ હાર પર કોંગ્રેસ અધ્યક...
3,"આ આંકડો માટે, અને વજન ઘટાડવા માટે પ્રકાશનનો દિ..."
4,આ ઠેકાઓ પરથી લીમડી તેમજ ઝાલોદના બૂટલેગરો વિદેશ...


### Date and time identifier 

* Gujarati Digits (0-9): Fall within the Unicode range \u0ae6 to \u0aef.

* Gujarati Letters (Months): All Gujarati characters fall within the Unicode block \u0A80 to \u0AFF.

In [13]:
gujarati_unicode_map = {
    "0": "\u0ae6",
    "1": "\u0ae7",
    "2": "\u0ae8",
    "3": "\u0ae9",
    "4": "\u0aea",
    "5": "\u0aeb",
    "6": "\u0aec",
    "7": "\u0aed",
    "8": "\u0aee",
    "9": "\u0aef"
}

In [ ]:
def extract_guj_date(text):
    num_pattern = r"[\u0ae6-\u0aef]{1,2}[\-\/\.][\u0ae6-\u0aef]{1,2}[\-\/\.][\u0ae6-\u0aef]{2,4}"
    text_pattern = r"[\u0ae6-\u0aef]{1,2}\s+[\u0a80-\u0aff]+\s+[\u0ae6-\u0aef]{2,4}"
    combine_pattern = r"{num_pattern}|{text_pattern}"
    return re.findall(num_pattern, text)


# Matches HH:MM or HH:MM:SS 
def extract_guj_time(text):
    time_pattern = r"[\u0ae6-\u0aef]{1,2}:[\u0ae6-\u0aef]{2}(?::[\u0ae6-\u0aef]{2})?"
    return re.findall(time_pattern, text)


In [ ]:
print(extract_guj_date("મારી ફ્લાઇટ ૧૫/૦૮/૨૦૨૬ ના રોજ ઉપડશે.છેલ્લું બિલ ૨૫.૧૨.૨૦૨૫ ના રોજ ચૂકવવામાં આવ્યું હતું.ગાંધીજીનો જન્મ ૦૨ ઓક્ટોબર ૧૮૬૯ માં થયો હતો."))

['૧૫/૦૮/૨૦૨૬', '૨૫.૧૨.૨૦૨૫']


In [ ]:
sample_time_text=sample_text = """
    મીટીંગ સવારે ૧૦:૩૦ કલાકે શરૂ થશે.
    ટાઈમર ૦૦:૧૫:૩૦ પર સેટ કરો.
    મારી બસ સાંજે ૫:૪૫ વાગ્યે ઉપડશે.
    """
print(extract_guj_time(sample_time_text))

['૧૦:૩૦', '૦૦:૧૫:૩૦', '૫:૪૫']


### Decimal number 

In [21]:
def extract_guj_num(text):
    decimal_pattern=r"[\u0ae6-\u0aef,]+\.[\u0ae6-\u0aef]+"
    return re.findall(decimal_pattern, text)


In [22]:
sample_decimal_text = """
    આ પુસ્તકની કિંમત ૧૫૦.૫૦ રૂપિયા છે.
    મારું વજન ૬૫.૭૫ કિલોગ્રામ છે.
    તેને પરીક્ષામાં ૮૯.૯૯ ટકા મળ્યા.

    """
print(extract_guj_num(sample_decimal_text))

['૧૫૦.૫૦', '૬૫.૭૫', '૮૯.૯૯']


### Email and url extracter

* pattern is divided into four main parts: [Prefix] [Domain] . [Extension] [Optional Path]
* (?: ... ) is a "non-capturing group." It groups things together but tells Python not to extract this part separately.


In [33]:
# .%+- - for std email symbols
def extract_guj_email(text):
    email_pattern = r"[\w\u0a80-\u0aff.%+-]+@[\w\u0a80-\u0aff.-]+\.[\w\u0a80-\u0aff]{2,}"
    return re.findall(email_pattern, text)

def extract_guj_urls(text):
    url_pattern = r"(?:https?:\/\/|www\.)[\w\u0a80-\u0aff.-]+\.[\w\u0a80-\u0aff]{2,}(?:\/[\w\u0a80-\u0aff.%+-]*)*"
    return re.findall(url_pattern, text)

In [34]:
sample_email_text = """
નમસ્કાર! જો તમને કોઈ પ્રશ્ન હોય, તો તમે અમને support@example.com પર ઈમેલ કરી શકો છો. 
સ્થાનિક ભાષામાં મદદ માટે, કૃપા કરીને સીધો સંપર્ક@ગુજરાત.ભારત પર મેલ મોકલો. 

અમારી નવી યોજનાઓ વિશે જાણવા માટે https://www.google.com/search જુઓ. 
સરકારી સુવિધાઓની નોંધણી માટેની સત્તાવાર વેબસાઈટ https://ગુજરાત.સરકાર.ભારત/લોગીન છે. 
તમે www.મારો-બ્લોગ.કોમ પર પણ અમારા નવા લેખો વાંચી શકો છો.
"""
sample_urls_text = """
નમસ્કાર! જો તમને કોઈ પ્રશ્ન હોય, તો તમે અમને support@example.com પર ઈમેલ કરી શકો છો. 
સ્થાનિક ભાષામાં મદદ માટે, કૃપા કરીને સીધો સંપર્ક@ગુજરાત.ભારત પર મેલ મોકલો. 

અમારી નવી યોજનાઓ વિશે જાણવા માટે https://www.google.com/search જુઓ. 
સરકારી સુવિધાઓની નોંધણી માટેની સત્તાવાર વેબસાઈટ https://ગુજરાત.સરકાર.ભારત/લોગીન છે. 
તમે www.મારો-બ્લોગ.કોમ પર પણ અમારા નવા લેખો વાંચી શકો છો.
"""

In [35]:
print(extract_guj_email(sample_email_text))

['support@example.com', 'સંપર્ક@ગુજરાત.ભારત']


In [36]:
print(extract_guj_urls(sample_urls_text))

['https://www.google.com/search', 'https://ગુજરાત.સરકાર.ભારત/લોગીન', 'www.મારો-બ્લોગ.કોમ']


### Sentence tokenizer


In [74]:
#to view full length text
with pd.option_context('display.max_colwidth', None):
    display(df[:4])

,text
0,આ વીડિયો જુઓ: ઊંઝા માર્કેટયાર્ડ આજથી 25 જુલાઈ સુધી બંધ
1,મિથેનોલ આવ્યો ક્યાંથી?
2,"આખરે ત્રણ રાજ્યોમાં મળેલ હાર પર કોંગ્રેસ અધ્યક્ષ રાહુલ ગાંધી દ્વારા પ્રથમ પ્રતિક્રિયા આપવામાં આવી છે. તેમણે કહ્યું કે, ત્રિપુરા, નાગાલેન્ડ અને મેઘાલયમાં લોકોના જનાદેશનો સ્વાગત કરીએ છે અને આ ક્ષેત્રના લોકોનો વિશ્વાસ ફરીથી જીતીવા માટે પ્રતિબદ્ધ છીએ."
3,"આ આંકડો માટે, અને વજન ઘટાડવા માટે પ્રકાશનનો દિવસ વિતાવવો ઉપયોગી છે, ઉદાહરણ તરીકે, અઠવાડિયામાં એક વખત. તમારા માટે એક વિકલ્પ પસંદ કરો જે અગવડતાને કારણે નહીં કરે. સૌથી વધુ લોકપ્રિય કીફિર પર અનલોડ છે."


In [75]:
# protected_items = [] store ump@12.com or 1.2 or 1/25/2560 etc
def mask_match(match,protected_items): 
        item=match.group(0) # to get full matched text not group of it
        protected_items.append(item) # to add into protected items
        return f"__PROTECTED_{len(protected_items)-1}__"

In [76]:
def unmask_text(masked_sentences,protected_items):
    final_sentence=[]
    for sentence in masked_sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        for idx,original_item in enumerate(protected_items):
            coded_pattern = f"__PROTECTED_{idx}__"
            if coded_pattern in sentence:
                sentence = sentence.replace(coded_pattern, original_item)
        final_sentence.append(sentence)
    return final_sentence

In [78]:
def mask_text(text:str):
    if not isinstance(text, str):
            return []
    
    CHAR_BLOCK = r"\w\u0a80-\u0aff"
    GUJ_DIGIT = r"\u0ae6-\u0aef"
    
    email_pattern = rf"[{CHAR_BLOCK}.%+-]+@[{CHAR_BLOCK}.-]+\.[{CHAR_BLOCK}]{{2,}}"
    url_pattern = rf"(?:https?:\/\/|www\.)[{CHAR_BLOCK}.-]+\.[{CHAR_BLOCK}]{{2,}}(?:\/[{CHAR_BLOCK}.%+-]*)*"
    decimal_pattern = rf"[{GUJ_DIGIT}]+\.[{GUJ_DIGIT}]+"
    time_pattern = rf"[{GUJ_DIGIT}]{{1,2}}:[{GUJ_DIGIT}]{{2}}(?::[{GUJ_DIGIT}]{{2}})?"
    date_pattern = rf"[{GUJ_DIGIT}]{{2}}[./-][{GUJ_DIGIT}]{{2}}[./-][{GUJ_DIGIT}]{{2,4}}"
    
    protection_pattern = f"({url_pattern}|{email_pattern}|{date_pattern}|{time_pattern}|{decimal_pattern})"
    protected_items=[]
    masked_text = re.sub(
        protection_pattern, 
        lambda m: mask_match(m, protected_items), 
        text
    )

    return masked_text,protected_items # return full text containng __PROTECTED_{} terms

In [79]:
def normalize_ellipses(text: str) -> str:
    if not isinstance(text, str):
        return ""
        
    # Pattern explanation:
    # \.{2,}  -> Matches 2 or more consecutive ASCII dots (.., ..., .....)
    # |       -> OR
    # \u2026  -> Matches the official single-character Unicode ellipsis (…)
    pattern = r'\.{2,}|\u2026'
    
    # Replace any match with a single dot
    return re.sub(pattern, '.', text)

* ### preprocessing step

In [80]:
def normalize_text(text):
    text=normalize_ellipses(text)
    
    return text
    

In [81]:

#    regex-based splitter on common sentence terminators (|, ?, !, newlines).
import re

def sentence_tokenize(text: str) -> list[str]:
     
     # to mask all matched patterns
    
    # ?<= use for look prev char of current loc.
    # [...] are list of character
    # \s+ for one or more spaces
    norm_text = normalize_text(text)
    masked_text,protected_items = mask_text(norm_text)

    split_pattern = r'(?<=[.?!\u0964|])\s+|\n+'

    masked_sentences = re.split(split_pattern,masked_text) # to get mask text 
    
    unmasked_sentences = unmask_text(masked_sentences,protected_items) # to get original sentences
    
    
    return unmasked_sentences


In [82]:

sample = df['text'].iloc[5]
sents = sentence_tokenize(sample)
print(f"Sentence count in given row: {len(sents)}")
for i, s in enumerate(sents[:5], 1):
    print(f"  [{i}] {s}")

Sentence count in given row: 2
  [1] કેન્દ્રીય મંત્રી જ્યોતિરાદિત્ય સિંધિયાની જન આશિર્વાદ યાત્રા દરમિયાન ઈન્દોરના એક પોલીસ સ્ટેશનમાં પ્રાણી ક્રૂરતાનો કેસ નોંધવામાં આવ્યો છે.
  [2] ભાજપના કાર્યકરોએ જન આશીર્વાદ યાત્રા દરમિયાન.


### Word Tokenizer

In [83]:
# 8. Gujarati stopword removal
GUJARATI_STOPWORDS = {
    # common function words in Gujarati (curated list)
    "છે", "છો", "છું", "હતો", "હતી", "હતું", "હતા", "થયું", "થયા", "થઈ", "થાય",
    "અને", "કે", "તો", "જો", "જે", "તે", "આ", "એ", "આમ", "એમ", "પણ", "પર",
    "માં", "ના", "ની", "નું", "ને", "થી", "સુધી", "માટે", "સાથે", "વિના",
    "હું", "તું", "તમે", "અમે", "આપણે", "તેઓ", "આના", "તેના",
    "કરે", "કરી", "કર્યું", "કરવા", "કરવામાં", "આવે", "આવી", "આવ્યું",
    "શકે", "શકો", "જોઈએ", "હોય", "છું", "છો", "છીએ",
    "આજે", "ગઈ", "ગયા", "ગયું", "કાલે", "પછી", "પહેલા", "હવે", "ત્યારે",
    "કોઈ", "કોઈક", "કંઈક", "શું", "કેમ", "ક્યાં", "ક્યારે", "કેટલું",
    "હા", "ના", "નહીં", "ન", "બહુ", "થોડું", "બધું", "બધા", "બધી",
    "અહીં", "ત્યાં", "જ્યાં", "જ્યારે", "જોકે", "કારણ", "કેમ કે",
    "the", "a", "an", "is", "are", "was", "were", "in", "on", "at",
    "to", "of", "and", "or", "for", "with", "by", "from", "as",
}

def remove_stopwords(tokens):
    return [t for t in tokens if t not in GUJARATI_STOPWORDS]

In [84]:
# 5. Word tokenization
#    Splits on whitespace and common punctuation. Keeps Gujarati script intact.

def word_tokenize(sentence: str) -> list[str]:
    if not isinstance(sentence, str):
        return []

    mask_sentence,protected_items = mask_text(sentence)

    pattern = r"[\s,;:/\"'()\[\]{}\\<>।!?\.,\-\"“”‘’«»]+" # all punctution marks
    raw_tokens = re.split(pattern, mask_sentence) # token of protected word
    
    # Clean up any empty strings created by re.split before unmasking
    clean_tokens = [t.strip() for t in raw_tokens if t.strip()]

    unmask_words = unmask_text(clean_tokens,protected_items)

    # stopword remover
    reduce_tokens = remove_stopwords(unmask_words)
    
    print(f"Before stopword removal: {len(tokens)} tokens")
    print(f"After  stopword removal: {len(reduce_tokens)} tokens")
    print("\nFirst 20 cleaned tokens:")
    print(reduce_tokens[:20])

    return list(unmask_words)




In [85]:
tokens = word_tokenize(sample)
print(f"Word count in first row: {len(tokens)}")
print(tokens[:20])

Before stopword removal: 24 tokens
After  stopword removal: 23 tokens

First 20 cleaned tokens:
['કેન્દ્રીય', 'મંત્રી', 'જ્યોતિરાદિત્ય', 'સિંધિયાની', 'જન', 'આશિર્વાદ', 'યાત્રા', 'દરમિયાન', 'ઈન્દોરના', 'એક', 'પોલીસ', 'સ્ટેશનમાં', 'પ્રાણી', 'ક્રૂરતાનો', 'કેસ', 'નોંધવામાં', 'આવ્યો', 'ભાજપના', 'કાર્યકરોએ', 'જન']
Word count in first row: 24
['કેન્દ્રીય', 'મંત્રી', 'જ્યોતિરાદિત્ય', 'સિંધિયાની', 'જન', 'આશિર્વાદ', 'યાત્રા', 'દરમિયાન', 'ઈન્દોરના', 'એક', 'પોલીસ', 'સ્ટેશનમાં', 'પ્રાણી', 'ક્રૂરતાનો', 'કેસ', 'નોંધવામાં', 'આવ્યો', 'છે', 'ભાજપના', 'કાર્યકરોએ']


In [86]:
# 6. Lowercasing (also covers Gujarati: there is no case, but it normalises)
lower_sample = sample.lower()
print("Original :", sample[:120])
print("Lowercased:", lower_sample[:120])

Original : કેન્દ્રીય મંત્રી જ્યોતિરાદિત્ય સિંધિયાની જન આશિર્વાદ યાત્રા દરમિયાન ઈન્દોરના એક પોલીસ સ્ટેશનમાં પ્રાણી ક્રૂરતાનો કેસ નોં
Lowercased: કેન્દ્રીય મંત્રી જ્યોતિરાદિત્ય સિંધિયાની જન આશિર્વાદ યાત્રા દરમિયાન ઈન્દોરના એક પોલીસ સ્ટેશનમાં પ્રાણી ક્રૂરતાનો કેસ નોં


In [87]:
# 7. Punctuation removal
PUNCT_RE = re.compile(r"[\u0964\u0965"          # Gujarati danda, double danda
                      r"!\"#\$%&'()*+,\-./:;<=>?@\[\\\]^_`{|}~।]",
                      flags=re.UNICODE)

def remove_punctuation(text: str) -> str:
    return PUNCT_RE.sub(" ", text) if isinstance(text, str) else ""

no_punct = remove_punctuation(sample)
print("Without punctuation:")
print(no_punct[:200])

Without punctuation:
કેન્દ્રીય મંત્રી જ્યોતિરાદિત્ય સિંધિયાની જન આશિર્વાદ યાત્રા દરમિયાન ઈન્દોરના એક પોલીસ સ્ટેશનમાં પ્રાણી ક્રૂરતાનો કેસ નોંધવામાં આવ્યો છે  ભાજપના કાર્યકરોએ જન આશીર્વાદ યાત્રા દરમિયાન   


In [ ]:
# 9. Full preprocessing pipeline applied to a small sample
def preprocess(text: str) -> list[str]:
    text = text.lower()
    text = remove_punctuation(text)
    toks = word_tokenize(text)
    toks = remove_stopwords(toks)
    return toks

sample_rows = df['text'].head(3).tolist()
for i, row in enumerate(sample_rows, 1):
    print(f"--- Row {i} ---")
    print("Original    :", row[:120])
    print("Preprocessed:", preprocess(row)[:15])
    print()

In [91]:
# 10. Vocabulary statistics on a small sample
from collections import Counter

sample_df = df.head(1000)            # 1k rows for speed
all_tokens = []
for t in sample_df['text']:
    all_tokens.extend(preprocess(t))

vocab = Counter(all_tokens)
print(f"Sample size       : {len(sample_df):,} rows")
print(f"Total tokens      : {len(all_tokens):,}")
print(f"Vocabulary size   : {len(vocab):,}")
print("\nTop 20 most common tokens:")
for word, freq in vocab.most_common(20):
    print(f"  {word:20s} {freq}")

Before stopword removal: 24 tokens
After  stopword removal: 8 tokens

First 20 cleaned tokens:
['વીડિયો', 'જુઓ', 'ઊંઝા', 'માર્કેટયાર્ડ', 'આજથી', '25', 'જુલાઈ', 'બંધ']
Before stopword removal: 24 tokens
After  stopword removal: 3 tokens

First 20 cleaned tokens:
['મિથેનોલ', 'આવ્યો', 'ક્યાંથી']
Before stopword removal: 24 tokens
After  stopword removal: 28 tokens

First 20 cleaned tokens:
['આખરે', 'ત્રણ', 'રાજ્યોમાં', 'મળેલ', 'હાર', 'કોંગ્રેસ', 'અધ્યક્ષ', 'રાહુલ', 'ગાંધી', 'દ્વારા', 'પ્રથમ', 'પ્રતિક્રિયા', 'આપવામાં', 'તેમણે', 'કહ્યું', 'ત્રિપુરા', 'નાગાલેન્ડ', 'મેઘાલયમાં', 'લોકોના', 'જનાદેશનો']
Before stopword removal: 24 tokens
After  stopword removal: 24 tokens

First 20 cleaned tokens:
['આંકડો', 'વજન', 'ઘટાડવા', 'પ્રકાશનનો', 'દિવસ', 'વિતાવવો', 'ઉપયોગી', 'ઉદાહરણ', 'તરીકે', 'અઠવાડિયામાં', 'એક', 'વખત', 'તમારા', 'એક', 'વિકલ્પ', 'પસંદ', 'કરો', 'અગવડતાને', 'કારણે', 'સૌથી']
Before stopword removal: 24 tokens
After  stopword removal: 55 tokens

First 20 cleaned tokens:
['ઠેકાઓ', 'પરથી', 'લીમડ